# Enhanced Market Basket Analysis

Walkthrough of the full MBA toolkit (keeps pace with `src/mba/` and the Streamlit app).

**vs the original `Association Analysis.ipynb`**
1. True **Apriori** (not brute-force combinations on orders of exact size 4)
2. **FP-Growth** and **Eclat** as complementary miners + runtime benchmark
3. **Association rules** with confidence, lift, leverage, conviction, Zhang, utility, aisle/dept scope
4. **PMI / Jaccard** affinity network
5. **Sequential** next-order transitions (`orders.csv`)
6. **Recommenders**: item-item CF + TruncatedSVD
7. Pointer to **Streamlit**: `streamlit run app/streamlit_app.py`

Default data: `data/sample/`. Drop real Instacart CSVs into `data/raw/` to switch automatically.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('.').resolve()
for candidate in [Path('.').resolve(), Path('..').resolve()]:
    if (candidate / 'src' / 'mba').exists():
        ROOT = candidate
        break

sys.path.insert(0, str(ROOT / 'src'))

from mba.affinity import pairwise_pmi
from mba.apriori import apriori, frequent_itemsets_to_frame
from mba.benchmark import benchmark_miners
from mba.data import (
    enrich_products,
    filter_frequent_products,
    filter_orders_by_size,
    load_order_products,
    load_orders,
    load_products,
    product_margin_map,
    product_name_map,
    resolve_dataset_dir,
    transactions_from_orders,
)
from mba.eclat import eclat
from mba.fp_growth import fp_growth
from mba.recommend import item_item_recommendations, svd_recommendations
from mba.rules import annotate_rule_departments, association_rules
from mba.sequential import sequential_transitions, user_sequences_from_orders
from mba.visualize import (
    plot_cooccurrence_network,
    plot_itemset_support,
    plot_metric_comparison,
    plot_pmi_heatmap_pairs,
    plot_rules_scatter,
    plot_top_products,
)

pd.set_option('display.max_colwidth', 120)
print('Project root:', ROOT)

## 1. Load & preprocess

Prefer `data/raw/` when present; otherwise use the sample generator output.

In [ ]:
base = resolve_dataset_dir(ROOT)
products = load_products(base / 'products.csv')
aisles = pd.read_csv(base / 'aisles.csv') if (base / 'aisles.csv').exists() else None
departments = pd.read_csv(base / 'departments.csv') if (base / 'departments.csv').exists() else None
products = enrich_products(products, aisles, departments)
name_map = product_name_map(products)
margin_map = product_margin_map(products)

train = load_order_products(base / 'order_products__train.csv')
prior = load_order_products(base / 'order_products__prior.csv') if (base / 'order_products__prior.csv').exists() else None
orders = load_orders(base / 'orders.csv') if (base / 'orders.csv').exists() else None

mine_src = prior if prior is not None else train
print('Dataset:', base)
print(f'Mining rows: {len(mine_src):,} | products catalog: {len(products):,}')
mine_src.head()

In [ ]:
MIN_COUNT = 30
MIN_SUPPORT = 0.01
MAX_LEN = 4

compacted, kept = filter_frequent_products(mine_src, min_count=MIN_COUNT)
compacted = filter_orders_by_size(compacted, min_size=2)
transactions = transactions_from_orders(compacted)
print(f'Kept products: {len(kept):,} | baskets: {len(transactions):,}')

## 2. Exploratory view

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
plot_top_products(compacted, name_map, top_n=15, ax=ax)
plt.show()

## 3. Apriori, FP-Growth, Eclat

In [ ]:
freq_a = apriori(transactions, min_support=MIN_SUPPORT, max_len=MAX_LEN)
freq_f = fp_growth(transactions, min_support=MIN_SUPPORT, max_len=MAX_LEN)
freq_e = eclat(transactions, min_support=MIN_SUPPORT, max_len=MAX_LEN)
apriori_df = frequent_itemsets_to_frame(freq_a, name_map=name_map)
fp_df = frequent_itemsets_to_frame(freq_f, name_map=name_map)

print('Same itemsets?', set(freq_a) == set(freq_f) == set(freq_e))
print('Counts:', len(freq_a), len(freq_f), len(freq_e))
timing = benchmark_miners(transactions, min_support=MIN_SUPPORT, max_len=MAX_LEN)
timing

In [ ]:
# Original project goal: top collections of four items
apriori_df[apriori_df['size'] == 4].head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_itemset_support(apriori_df, size=4, top_n=8, ax=axes[0])
plot_metric_comparison(apriori_df, fp_df, ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Association rules (Zhang, utility, scope)

In [ ]:
rules = association_rules(
    freq_a,
    min_confidence=0.2,
    min_lift=1.05,
    name_map=name_map,
    margin_map=margin_map or None,
)
rules = annotate_rule_departments(rules, products)
print(rules['scope'].value_counts() if 'scope' in rules.columns else 'no scope')
rules.head(12)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_rules_scatter(rules, top_n=50, ax=ax)
plt.show()

## 5. PMI affinity

In [ ]:
pmi = pairwise_pmi(transactions, min_count=8, top_n=40, name_map=name_map)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_pmi_heatmap_pairs(pmi, top_n=12, ax=axes[0])
plot_cooccurrence_network(pmi, top_edges=22, ax=axes[1])
plt.tight_layout()
plt.show()
pmi.head(10)

## 6. Sequential next-order transitions

In [ ]:
if orders is None:
    print('No orders.csv — skip sequential section (regenerate sample data).')
else:
    seqs = user_sequences_from_orders(orders, mine_src)
    transitions = sequential_transitions(
        seqs, min_support=0.008, min_confidence=0.12, name_map=name_map, top_n=20
    )
    display(transitions)

## 7. Recommenders

In [ ]:
seed = int(compacted['product_id'].value_counts().index[0])
print('Seed:', name_map.get(seed, seed))
cf = item_item_recommendations(transactions, seed, top_n=8, name_map=name_map)
svd = svd_recommendations(transactions, [seed], top_n=8, name_map=name_map)
print('Item-item CF'); display(cf)
print('SVD'); display(svd)

## 8. Streamlit demo

For the interactive lab (sliders, tabs, Plotly charts):

```bash
streamlit run app/streamlit_app.py
```

Or open the pre-exported HTML under `figures/interactive/`.

## Takeaways

| Technique | Strength | Use when |
|---|---|---|
| **Apriori** | Interpretable prune | Teaching / reference |
| **FP-Growth / Eclat** | Faster | Production frequent-itemset mining |
| **Rules (+ Zhang / utility / scope)** | Actionable cross-sell | Merchandising / bundles |
| **PMI network** | Surprising affinities | Niche co-occurrence |
| **Sequential** | Time-aware habits | Next-order suggestions |
| **CF / SVD** | Personalized neighbors | Seed-product recommendations |

On the sample data, the sparkling-water and berry 4-item themes from the original notebook still rank first — now with proper mining, metrics, and visuals.